# A DEMO OF TRANSFORMER for TEXT_SUMMARIZATION:

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

# --- 1. Positional Embedding Layer ---
# This layer is crucial for Transformers as it provides information about the
# order of the words in the sequence. Without it, the model would be
# permutation-invariant and would not understand word order.
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super(PositionalEmbedding, self).__init__(**kwargs)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        # Word embedding layer
        self.word_embedding = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        # Positional embedding layer
        self.position_embedding = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )

    def call(self, inputs):
        # The position indices will be a sequence from 0 to sequence_length-1
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        # Get word embeddings
        embedded_words = self.word_embedding(inputs)
        # Get positional embeddings
        embedded_positions = self.position_embedding(positions)
        # Add the two embeddings together
        return embedded_words + embedded_positions

# --- 2. Transformer Encoder Block ---
# The encoder block processes the input sequence and creates a contextualized
# representation of it.
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim)]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        # Self-attention with residual connection and layer normalization
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        # Feed-forward network with residual connection and layer normalization
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# --- 3. Transformer Decoder Block ---
# The decoder block is responsible for generating the output sequence one token
# at a time. It uses a masked self-attention layer and a cross-attention layer.
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerDecoder, self).__init__(**kwargs)
        # Masked self-attention layer
        self.att1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        # Cross-attention layer (attends to the encoder output)
        self.att2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim)]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
        self.dropout3 = layers.Dropout(rate)

    def call(self, enc_output, dec_input, training=False):
        # Generate a causal mask for the decoder self-attention
        # This prevents the decoder from "cheating" and looking at future tokens
        # in the output sequence.
        causal_mask = self.get_causal_attention_mask(dec_input)

        # Masked self-attention
        attn1_output = self.att1(dec_input, dec_input, attention_mask=causal_mask)
        attn1_output = self.dropout1(attn1_output, training=training)
        out1 = self.layernorm1(dec_input + attn1_output)

        # Cross-attention to encoder output
        attn2_output = self.att2(out1, enc_output)
        attn2_output = self.dropout2(attn2_output, training=training)
        out2 = self.layernorm2(out1 + attn2_output)

        # Feed-forward network
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        return self.layernorm3(out2 + ffn_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        return tf.tile(mask, [batch_size, 1, 1])


# --- 4. The Full Transformer Model for Summarization ---
# This model combines the encoder and decoder to perform the Seq2Seq task.
class TextSummarizerTransformer(Model):
    def __init__(
        self,
        embed_dim,
        num_heads,
        ff_dim,
        input_vocab_size,
        target_vocab_size,
        input_maxlen,
        target_maxlen,
        rate=0.1,
    ):
        super(TextSummarizerTransformer, self).__init__()
        self.input_embedder = PositionalEmbedding(input_maxlen, input_vocab_size, embed_dim)
        self.target_embedder = PositionalEmbedding(target_maxlen, target_vocab_size, embed_dim)
        self.encoder_block = TransformerEncoder(embed_dim, num_heads, ff_dim, rate)
        self.decoder_block = TransformerDecoder(embed_dim, num_heads, ff_dim, rate)
        self.final_dense = layers.Dense(target_vocab_size)

    def call(self, inputs, training=False):
        # Unpack the inputs (encoder input and decoder input)
        encoder_input, decoder_input = inputs

        # Encode the input sequence
        encoder_embedded = self.input_embedder(encoder_input)
        encoder_output = self.encoder_block(encoder_embedded, training=training)

        # Decode the target sequence
        decoder_embedded = self.target_embedder(decoder_input)
        decoder_output = self.decoder_block(encoder_output, decoder_embedded, training=training)

        # Pass the decoder output through the final dense layer to get
        # the vocabulary logits
        final_output = self.final_dense(decoder_output)

        return final_output

# --- 5. Toy Example and Model Instantiation ---

if __name__ == "__main__":
    # Define hyperparameters (feel free to adjust)
    embed_dim = 64
    num_heads = 4
    ff_dim = 128
    input_maxlen = 10  # small input text length
    target_maxlen = 20 # large output text length
    vocab_size = 5000  # including start and end tokens

    # We need a start and end token. Let's assume their IDs are known.
    # In a real scenario, you'd get these from your tokenizer.
    start_token_id = 1
    end_token_id = 2

    # Create dummy data for demonstration
    # Shape: (batch_size, sequence_length)
    # The `+1` is for the special tokens.
    dummy_encoder_input = np.random.randint(
        3, vocab_size, size=(32, input_maxlen)
    )

    # The decoder input will be the target sequence shifted to the right,
    # with a start token at the beginning.
    dummy_decoder_input = np.random.randint(
        3, vocab_size, size=(32, target_maxlen - 1)
    )
    start_tokens = np.full((32, 1), start_token_id)
    dummy_decoder_input = np.concatenate([start_tokens, dummy_decoder_input], axis=1)

    # The target output will be the decoder input shifted to the left,
    # with an end token at the end.
    dummy_target_output = np.random.randint(
        3, vocab_size, size=(32, target_maxlen - 1)
    )
    end_tokens = np.full((32, 1), end_token_id)
    dummy_target_output = np.concatenate([dummy_target_output, end_tokens], axis=1)

    # Instantiate the model
    model = TextSummarizerTransformer(
        embed_dim=embed_dim,
        num_heads=num_heads,
        ff_dim=ff_dim,
        input_vocab_size=vocab_size,
        target_vocab_size=vocab_size,
        input_maxlen=input_maxlen,
        target_maxlen=target_maxlen,
    )

    # Compile the model
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # The model expects a tuple of inputs: (encoder_input, decoder_input)
    # The target is the true next token at each time step
    model.fit(
        x=(dummy_encoder_input, dummy_decoder_input),
        y=dummy_target_output,
        epochs=1
    )

    # Print the model summary
    model.summary()

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - accuracy: 0.0000e+00 - loss: 15.6171


Model: "text_summarizer_transformer_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ positional_embedding_2          │ ?                      │       320,640 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding_3          │ ?                      │       321,280 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_1           │ ?                      │        83,200 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_1           │ ?                      │       149,696 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (32, 20, 5000)         │       325,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,599,450 (13.73 MB)

 Trainable params: 1,199,816 (4.58 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,399,634 (9.15 MB)

In [7]:
import numpy as np

# This script assumes you have a pre-defined tokenizer and vocabulary, and have
# already tokenized and padded your input and target sequences.

# --- 1. Define dummy data for demonstration ---
# In a real-world scenario, you would replace these with your actual
# tokenized and padded data.

# Assume these are the special token IDs from your tokenizer
start_token_id = 1
end_token_id = 2
vocab_size = 5000

# Let's say your padded input data has a max length of 10
input_maxlen = 10

# Assume this is your padded input text data, ready for the encoder.
# This would be the result of tokenizing and padding your source texts.
input_padding = np.random.randint(3, vocab_size, size=(32, input_maxlen))

# Assume this is your padded target summary data, with a max length of 20.
# This would be the result of tokenizing and padding your target summaries.
target_maxlen = 20
padded_target_summaries = np.random.randint(3, vocab_size, size=(32, target_maxlen))

# --- 2. Prepare the data for the Transformer's Decoder ---

# The decoder_input_data needs to have a START token at the beginning.
# It's created by taking the target data and prepending the START token,
# then removing the last token to maintain the same sequence length as the target.
start_tokens = np.full((padded_target_summaries.shape[0], 1), start_token_id)
decoder_input_data = np.concatenate([start_tokens, padded_target_summaries[:, :-1]], axis=1)

# model will be trained to predict at each step.
end_tokens = np.full((padded_target_summaries.shape[0], 1), end_token_id)
decoder_target_data = np.concatenate([padded_target_summaries[:, 1:], end_tokens], axis=1)

# --- How to use this data with your TextSummarizerTransformer model ---
# You can now pass these prepared datasets to your model's .fit() method:
#
model.fit(
x=(input_padding, decoder_input_data),
y=decoder_target_data,
epochs=20

)


Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.0000e+00 - loss: 15.3088
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy: 0.0000e+00 - loss: 14.3188
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.0031 - loss: 13.5505
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.0109 - loss: 12.9479
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.0203 - loss: 12.4799
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.0406 - loss: 12.0702
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step - accuracy: 0.0547 - loss: 11.6739
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.0656 - loss: 11.3187
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.0656 - loss: 11.0564
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.0594 - loss: 10.8217
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.0625 - loss: 10.6014
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - ac

In [ ]:
! pip install torch torchvision torchaudio


Defaulting to user installation because normal site-packages is not writeable
  Using cached torch-2.8.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-

In [3]:
from transformers import pipeline
classifier = pipeline('sentiment-analysis')

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [2]:
classifier('We are very happy to show you the 🤗 Transformers library.')

NameError: name 'classifier' is not defined